# 01 · Dataset Audit (v2 – calibrated on the real file)
## Renewable Energy Transition in France: Regional Dynamics and the Rise of Sustainable Electricity Production

**Project:** EFREI Paris – Data Storytelling (Streamlit) · #EFREIDataStories2026
**Author:** Soumahoro Souleymane
**Dataset:** `production-regionale-mensuelle-filiere.csv` – monthly electricity production by region and energy source (French open data, ODRE / data.gouv.fr)
**Research question:** *How are French regions progressing toward a renewable electricity future?*

### What changed compared with the first (generic) version
The first run on your machine gave real facts, so this version **stops guessing** and uses them:

| Finding of run 1 | What this version does |
|---|---|
| 11 columns, wide layout, unit = **GWh** (not MWh) | Columns, roles and units are declared explicitly (data dictionary). |
| Nuclear = 864 missing (46 %), flagged "High" | Tested whether it is a **real hole or a structural zero** (region without nuclear plant) → rule + score adjusted. |
| File weighs 21.5 MB for only 1 872 rows | Finds the cause (repeated polygon text) and splits it into a small *region dimension*. |
| `geo_shape_region` is a polygon as text | Parses and validates it → produces `regions.geojson` so the map needs **no external file**. |
| Global 3×IQR flagged ~214 "outliers" | Redone **per region**, because big regions are not anomalies. |
| 34 negative values | Listed row by row with their size. |
| Results were only read by eye | A **reconciliation table** compares every number with run 1 (PASS/FAIL) – reproducibility proof. |

### Audit plan
1 Loading & fingerprint · 2 Column inspection · 3 Data types · 4 Missing values · 5 Duplicates · 6 Temporal coverage · 7 Regional coverage (+ geometry) · 8 Energy sources · 9 Quality scorecard · 10 Reconciliation & exports · 11 Final synthesis (summary, issues, recommendations, features).

> This notebook **only audits**: the source file is never modified, and no Streamlit code is written here.

## 0 · Setup
**What:** imports, the schema of the file, the reference numbers of run 1, and the file location.
**Why:** a notebook must run from a clean checkout (graded under *Reproducibility*). The path list tries your Windows path first, then `data/` relative paths, so the same notebook works after zipping the project. Constants (schema, energy families, colours) live in one place so that notebook 02 and the app reuse exactly the same definitions.

In [ ]:
import os, re, csv, json, hashlib, unicodedata, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

DATA_NAME = "production-regionale-mensuelle-filiere.csv"
CANDIDATES = [
    os.environ.get("AUDIT_CSV", ""),
    r"C:\Users\HP\Desktop\EFREI\2026-2027\Data_anlysis\personnal_project_Efrei\data\production-regionale-mensuelle-filiere.csv",
    str(Path("data") / DATA_NAME),
    str(Path("..") / "data" / DATA_NAME),
    DATA_NAME,
]
DATA_PATH = next((Path(p) for p in CANDIDATES if p and Path(p).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("CSV not found. Tried:\n" + "\n".join(p for p in CANDIDATES if p))

OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)

# ---------- schema (snake_case names produced in section 2) ----------
SCHEMA = {
    "mois":                      ("time",     None,  "Month, text 'YYYY-MM'"),
    "code_insee_region":         ("id",       None,  "INSEE region code"),
    "region":                    ("category", None,  "Region name"),
    "production_nucleaire_gwh":  ("measure",  "GWh", "Nuclear electricity production"),
    "production_thermique_gwh":  ("measure",  "GWh", "Thermal (fossil) electricity production"),
    "production_hydraulique_gwh":("measure",  "GWh", "Hydro electricity production"),
    "production_eolienne_gwh":   ("measure",  "GWh", "Wind electricity production"),
    "production_solaire_gwh":    ("measure",  "GWh", "Solar electricity production"),
    "production_bioenergies_gwh":("measure",  "GWh", "Bioenergy electricity production"),
    "geo_shape_region":          ("geo",      None,  "Region boundary as GeoJSON geometry (text)"),
    "geo_point_region":          ("geo",      None,  "Region centroid as 'lat, lon' (text)"),
}
MEASURES = [c for c, v in SCHEMA.items() if v[0] == "measure"]
NUC = "production_nucleaire_gwh"

# Assumption (to be documented in the app): classification of sources
FAMILY = {
    "production_nucleaire_gwh":   "Nuclear",
    "production_thermique_gwh":   "Fossil / thermal",
    "production_hydraulique_gwh": "Renewable",
    "production_eolienne_gwh":    "Renewable",
    "production_solaire_gwh":     "Renewable",
    "production_bioenergies_gwh": "Renewable",
}
NEW_RENEWABLES = ["production_eolienne_gwh", "production_solaire_gwh", "production_bioenergies_gwh"]
LABEL = {c: c.replace("production_", "").replace("_gwh", "").capitalize() for c in MEASURES}
LABEL["production_eolienne_gwh"] = "Wind"; LABEL["production_nucleaire_gwh"] = "Nuclear"
LABEL["production_hydraulique_gwh"] = "Hydro"; LABEL["production_solaire_gwh"] = "Solar"
LABEL["production_thermique_gwh"] = "Thermal"; LABEL["production_bioenergies_gwh"] = "Bioenergy"
# colour-blind-safe palette (Okabe-Ito)
COLOR = {"production_nucleaire_gwh": "#E69F00", "production_thermique_gwh": "#7F7F7F",
         "production_hydraulique_gwh": "#0072B2", "production_eolienne_gwh": "#56B4E9",
         "production_solaire_gwh": "#D55E00", "production_bioenergies_gwh": "#009E73"}
FAM_COLOR = {"Renewable": "#009E73", "Nuclear": "#E69F00", "Fossil / thermal": "#7F7F7F"}

# The 13 metropolitan regions (post-2016 reform) with INSEE codes - used as a reference list
OFFICIAL_REGIONS = {
    "84": "Auvergne-Rhône-Alpes", "27": "Bourgogne-Franche-Comté", "53": "Bretagne",
    "24": "Centre-Val de Loire", "94": "Corse", "44": "Grand Est", "32": "Hauts-de-France",
    "11": "Île-de-France", "28": "Normandie", "75": "Nouvelle-Aquitaine", "76": "Occitanie",
    "52": "Pays de la Loire", "93": "Provence-Alpes-Côte d'Azur"}

# Numbers read in run 1 (your screenshot/PDF) -> checked again in section 10
RUN1 = {
    "rows": 1872, "columns": 11, "regions": 13, "months": 144, "missing_cells": 864,
    "nuclear_total_gwh": 4398501.37, "hydro_total_gwh": 743387.07, "thermal_total_gwh": 431166.08,
    "wind_total_gwh": 407731.99, "solar_total_gwh": 179541.21, "bio_total_gwh": 113547.44,
    "neg_nuclear": 6, "neg_thermal": 26, "neg_wind": 2, "renewable_share_cum_pct": 23.0,
}
ISSUES = []   # (severity, topic, finding, decision)

def issue(sev, topic, finding, decision):
    ISSUES.append((sev, topic, finding, decision))

print("Using file :", DATA_PATH)
print("Size (MB)  :", round(DATA_PATH.stat().st_size / 1e6, 2))

## 1 · Loading the raw file and fingerprint
**What:** read the CSV (encoding and separator are detected, not assumed), keep an untouched copy `RAW`, and compute a **SHA-256 fingerprint** of the file.
**Why:** run 1 found `utf-8-sig` and `;`. The fingerprint is stored in the report: anyone re-running the project can check that they audit the *same* file (a graded reproducibility point). A wrong separator would give a single huge column, so we also stop if fewer than 2 columns are found.

In [ ]:
def read_flex(path):
    last_err = None
    for enc in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            with open(path, "r", encoding=enc) as f:
                head = f.read(20000)
            try:
                sep = csv.Sniffer().sniff(head, delimiters=";,\t|").delimiter
            except csv.Error:
                sep = ";"
            return pd.read_csv(path, sep=sep, encoding=enc, low_memory=False), enc, sep
        except UnicodeDecodeError as e:
            last_err = e
    raise last_err

RAW, ENCODING, SEPARATOR = read_flex(DATA_PATH)
SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
assert RAW.shape[1] > 1, "Separator problem: file parsed into a single column."

print(f"Encoding : {ENCODING}   | Separator: {SEPARATOR!r}")
print(f"Shape    : {RAW.shape[0]:,} rows x {RAW.shape[1]} columns")
print(f"Memory   : {RAW.memory_usage(deep=True).sum()/1e6:.2f} MB   | File on disk: {DATA_PATH.stat().st_size/1e6:.2f} MB")
print(f"SHA-256  : {SHA256}")
RAW.head(5)

**How to read this:** 1 872 rows = **13 regions × 144 months**, which is a very small table. A 21 MB file for so few rows is suspicious: section 2 explains why.

In [ ]:
RAW.sample(6, random_state=42)    # fixed seed -> same sample on every machine

## 2 · Column inspection (data dictionary)
**What:** every column with its role, unit, type, number of non-null values, distinct values and an example. Column names are converted to `snake_case` (no accents/spaces) because accented names break code and Streamlit widgets; the original label is kept for chart titles.
**Why:** the course checklist asks for a *clean schema & metadata (data dictionary, units)*. The unit (**GWh per month per region**) is now confirmed by the column headers and must be written on every axis of the app.

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"[^0-9a-zA-Z]+", "_", s).strip("_").lower()

df = RAW.copy()
ORIG_LABEL = {norm(c): c for c in RAW.columns}
df.columns = [norm(c) for c in RAW.columns]

missing_cols = [c for c in SCHEMA if c not in df.columns]
extra_cols = [c for c in df.columns if c not in SCHEMA]
if missing_cols:
    raise KeyError(f"Expected columns not found: {missing_cols}\nFound: {list(df.columns)}")
print("Schema check: all", len(SCHEMA), "expected columns present.", "Unexpected extra columns:", extra_cols or "none")

col_audit = pd.DataFrame({
    "column": df.columns,
    "original_label": [ORIG_LABEL[c] for c in df.columns],
    "role": [SCHEMA[c][0] for c in df.columns],
    "unit": [SCHEMA[c][1] for c in df.columns],
    "dtype_raw": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "n_unique": df.nunique().values,
    "example": [str(df[c].dropna().iloc[0])[:40] for c in df.columns],
    "description": [SCHEMA[c][2] for c in df.columns],
})
col_audit

**How to read this:** `n_unique = 13` for code, region and the two geo columns means they only describe the **13 regions** – they repeat on every row. Measures have ~1 800 distinct values (real numbers). `production_nucleaire_gwh` has only 1 008 non-null values: studied in section 4.

**Why is the file 21 MB?** Let us measure where the memory goes.

In [ ]:
mem = df.memory_usage(deep=True).drop("Index").sort_values(ascending=False)
mem_tbl = pd.DataFrame({"MB": mem / 1e6, "share_%": mem / mem.sum() * 100})
display(mem_tbl.head(6))

geo_cols = [c for c in df.columns if SCHEMA[c][0] == "geo"]
geo_share = mem[geo_cols].sum() / mem.sum() * 100
print(f"\nThe 2 geo columns hold {geo_share:.1f}% of the memory, although they contain only {df[geo_cols[0]].nunique()} distinct values.")
if geo_share > 50:
    issue("Medium", "Performance",
          f"Polygon text repeated on every row: geo columns = {geo_share:.0f}% of memory ({mem.sum()/1e6:.1f} MB for {len(df):,} rows).",
          "Split into a fact table (region x month x source) and a 13-row region dimension holding the geometry.")

## 3 · Data types analysis
**What:** compare the *stored* type of each column with its *real* meaning and apply the fixes on the working copy `df` (never on `RAW`).
**Why:** run 1 showed that `mois` is text (`'2025-12'`): sorting works by luck but resampling, year/month extraction and date sliders would not. `code_insee_region` is an integer but is an **identifier**, not a quantity (nobody averages region codes) → text. Measures are already `float64` (no decimal-comma problem in this file).

In [ ]:
types = pd.DataFrame({"column": df.columns, "stored": df.dtypes.astype(str).values,
                      "expected": [{"time": "datetime (month start)", "id": "text (2 digits)", "category": "category/text",
                                    "measure": "float64", "geo": "parsed geometry / lat-lon"}[SCHEMA[c][0]] for c in df.columns]})
display(types)

# --- fixes on the working copy
df["date"] = pd.to_datetime(df["mois"], format="%Y-%m", errors="coerce")
n_bad_dates = int(df["date"].isna().sum())
df["code_insee_region"] = df["code_insee_region"].astype(str).str.zfill(2)
for c in MEASURES:
    assert pd.api.types.is_float_dtype(df[c]), f"{c} should already be numeric"

print(f"Unparseable months: {n_bad_dates}")
print("Region codes now text:", sorted(df['code_insee_region'].unique()))
issue("Low", "Types", "`mois` stored as text 'YYYY-MM'; INSEE code stored as integer.",
      "Parse `mois` to month-start datetime `date`; keep region code as 2-character text.")
if n_bad_dates:
    issue("High", "Types", f"{n_bad_dates} months could not be parsed.", "Inspect and fix before analysis.")

## 4 · Missing values analysis
**What:** count missing cells per column, then test **where** they are (which region, which month).
**Why:** run 1 reported 864 missing values, **all in nuclear production** (46 %). A naive tool calls this a "High" problem. But 864 = 6 regions × 144 months exactly: it suggests **6 regions never have a value** – i.e. they have **no nuclear plant**. If that is true it is not a data hole but a *structural zero*, and filling it with `0` is correct for shares; if instead the gaps were scattered, filling would be wrong. The code below proves which case we are in.

In [ ]:
miss = pd.DataFrame({"missing": df[MEASURES + ["region"]].isna().sum(),
                     "missing_%": (df[MEASURES + ["region"]].isna().mean() * 100).round(2)})
print(f"Total missing cells (11 original columns): {int(df[list(SCHEMA)].isna().sum().sum()):,}")
display(miss[miss.missing > 0] if (miss.missing > 0).any() else "No missing values")

# disguised missing values in text columns
DISG = {"", " ", "-", "--", "n/a", "na", "null", "none", "nd", "n.d.", "?"}
txt = ["mois", "region", "geo_shape_region", "geo_point_region"]
disg = {c: int(df[c].dropna().astype(str).str.strip().str.lower().isin(DISG).sum()) for c in txt}
print("Disguised missing values (placeholder strings):", {k: v for k, v in disg.items() if v} or "none")

In [ ]:
# Is nuclear missingness structural (all-or-nothing per region) or scattered?
nuc = df.groupby("region")[NUC].agg(rows="size", missing=lambda s: int(s.isna().sum()))
nuc["status"] = np.select([nuc.missing == nuc.rows, nuc.missing == 0], ["NEVER has a value (structural)", "always present"], "PARTIAL (real holes)")
display(nuc.sort_values("missing", ascending=False))

NO_NUCLEAR = sorted(nuc.index[nuc.missing == nuc.rows])
PARTIAL = sorted(nuc.index[(nuc.missing > 0) & (nuc.missing < nuc.rows)])
print(f"\nRegions with NO nuclear value at all ({len(NO_NUCLEAR)}): {NO_NUCLEAR}")
print(f"Regions with scattered nuclear gaps ({len(PARTIAL)}): {PARTIAL or 'none'}")

if not PARTIAL:
    issue("Info", "Missing values",
          f"Nuclear: {int(df[NUC].isna().sum()):,} missing cells ({df[NUC].isna().mean()*100:.1f}%) = {len(NO_NUCLEAR)} regions x {df['date'].nunique()} months, "
          "all-or-nothing per region -> structural absence (no nuclear plant), not a data hole.",
          "Treat as 0 GWh for totals/shares; keep boolean `has_nuclear` per region; never impute.")
else:
    issue("High", "Missing values", f"Nuclear has scattered gaps in {PARTIAL}.", "Investigate those region-months before any aggregation.")

In [ ]:
# Visual proof: % missing per region x measure
mm = df.groupby("region")[MEASURES].apply(lambda g: g.isna().mean() * 100)
fig, ax = plt.subplots(figsize=(8.5, 5.2))
im = ax.imshow(mm.values, aspect="auto", cmap="Reds", vmin=0, vmax=100)
ax.set_xticks(range(len(MEASURES))); ax.set_xticklabels([LABEL[c] for c in MEASURES])
ax.set_yticks(range(len(mm))); ax.set_yticklabels(mm.index)
for i in range(mm.shape[0]):
    for j in range(mm.shape[1]):
        ax.text(j, i, f"{mm.values[i, j]:.0f}", ha="center", va="center",
                color="white" if mm.values[i, j] > 50 else "black", fontsize=8)
ax.set_title("Missing values by region and source (% of 144 months)"); ax.grid(False)
fig.colorbar(im, label="% missing"); plt.tight_layout(); plt.show()

**How to read this:** every cell is 0 % or 100 %. Rows in dark red are regions without nuclear generation. There is **no scattered missingness** in any other source, so the dataset is, in practice, complete once this structural rule is applied. (Corsica, for instance, is an island grid with no nuclear plant.)

## 5 · Duplicate detection
**What:** (1) exact duplicate rows, (2) duplicate **keys** – the pair (month, region) must be unique in a monthly regional table, (3) conflicting duplicates (same key, different values).
**Why:** duplicates would double-count production and inflate regional comparisons. Run 1 found none; we re-check and keep the test in the notebook as a permanent validation.

In [ ]:
KEY = ["date", "region"]
n_full = int(df.duplicated().sum())
n_key = int(df.duplicated(KEY).sum())
n_code_key = int(df.duplicated(["date", "code_insee_region"]).sum())
print(f"Exact duplicate rows         : {n_full}")
print(f"Duplicate (month, region)    : {n_key}")
print(f"Duplicate (month, INSEE code): {n_code_key}")
if n_key:
    d = df[df.duplicated(KEY, keep=False)].sort_values(KEY)
    display(d.head(10))
    issue("High", "Duplicates", f"{n_key} duplicated (month, region) keys.", "Drop exact duplicates; document a rule for conflicting values.")
else:
    print("OK - the table is unique at (month, region) level: 13 x 144 = 1 872 rows.")

## 6 · Temporal coverage
**What:** first/last month, number of months, gaps, rows per year, and per-region continuity.
**Why:** the story is *progress over time*; comparisons are only fair if every region has the same, continuous, full-year timeline. A partial last year would bias annual totals.

In [ ]:
start, end = df["date"].min(), df["date"].max()
expected_months = pd.date_range(start, end, freq="MS")
observed_months = pd.DatetimeIndex(sorted(df["date"].unique()))
gap_months = expected_months.difference(observed_months)

print(f"Period : {start:%Y-%m} -> {end:%Y-%m}  ({len(expected_months)} months = {len(expected_months)/12:.0f} years)")
print(f"Months missing in the whole table: {len(gap_months)}")

per_year = df.assign(year=df["date"].dt.year).groupby("year").agg(rows=("date", "size"), months=("date", "nunique"), regions=("region", "nunique"))
display(per_year.T)
partial_years = per_year.index[per_year.months < 12].tolist()
print("Partial years:", partial_years or "none (2014-2025 are all complete)")

reg_t = df.groupby("region")["date"].agg(first="min", last="max", observed="nunique")
reg_t["expected"] = (reg_t["last"].dt.year - reg_t["first"].dt.year) * 12 + (reg_t["last"].dt.month - reg_t["first"].dt.month) + 1
reg_t["missing_months"] = reg_t["expected"] - reg_t["observed"]
reg_t["coverage_%"] = (reg_t["observed"] / reg_t["expected"] * 100).round(1)
display(reg_t)

BALANCED = bool(reg_t["missing_months"].eq(0).all() and reg_t["first"].nunique() == 1 and reg_t["last"].nunique() == 1)
print("Balanced panel (same start, same end, no gap):", BALANCED)
if gap_months.size or not BALANCED or partial_years:
    issue("High", "Temporal", "Gaps, unbalanced regions or partial years detected.", "Normalise before annual comparisons.")
else:
    issue("Info", "Temporal", "144 consecutive months (2014-01 to 2025-12), 13 regions, no gap, no partial year.",
          "Annual aggregation is safe; the last year (2025) is complete.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(per_year.index.astype(str), per_year["rows"], color="#0072B2")
ax.set_ylim(0, per_year["rows"].max() * 1.15)
for x, v in zip(per_year.index.astype(str), per_year["rows"]): ax.text(x, v + 2, str(v), ha="center", fontsize=8)
ax.set_xlabel("Year"); ax.set_ylabel("Rows (13 regions x 12 months)"); ax.set_title("Records per year - perfectly flat = balanced panel")
plt.tight_layout(); plt.show()

**How to read this:** all bars are 156 = 13 × 12. The timeline is regular, so a date slider and year-on-year comparisons will behave properly.

## 7 · Regional coverage and geometry
**What:** check the region list against the **13 official metropolitan regions**, check that code ↔ name is one-to-one, then **parse and validate the geometry columns**.
**Why:** (a) 2014–2015 predate the 2016 regional reform; if the file used old regions the series would break – here the 13 post-reform names are used for all years (good). (b) The brief requires *at least one map if geo fields exist*: the file ships its own polygons, so we can draw a map without GeoPandas or an external GeoJSON – but only if the geometry is valid.

In [ ]:
reg = df.groupby(["code_insee_region", "region"]).size().rename("rows").reset_index()
reg["share_%"] = (reg.rows / reg.rows.sum() * 100).round(2)
display(reg)

code_name_ok = bool(df.groupby("region")["code_insee_region"].nunique().eq(1).all() and df.groupby("code_insee_region")["region"].nunique().eq(1).all())
found = dict(zip(reg["code_insee_region"], reg["region"]))
unknown = {k: v for k, v in found.items() if OFFICIAL_REGIONS.get(k) != v}
absent = {k: v for k, v in OFFICIAL_REGIONS.items() if k not in found}
print("Code <-> name one-to-one       :", code_name_ok)
print("Matches the 13 official regions:", not unknown and not absent)
if unknown: print("  Different / unknown :", unknown)
if absent:  print("  Missing             :", absent)
print("Overseas regions (DROM) present:", any(c in found for c in ["01", "02", "03", "04", "06"]))
if code_name_ok and not unknown and not absent:
    issue("Info", "Regions", "13 metropolitan regions, post-2016 names for all years, code<->name one-to-one.",
          "No harmonisation needed. Scope = metropolitan France only (no overseas territories): state it in the app.")
else:
    issue("High", "Regions", f"Region mismatch. Unknown={unknown}, absent={absent}", "Harmonise labels before joins.")

In [ ]:
# ---------- geometry validation ----------
def walk(coords):
    """Yield all (lon, lat) points of any nested GeoJSON coordinate list."""
    if coords and isinstance(coords[0], (int, float)):
        yield coords[0], coords[1]
    else:
        for c in coords:
            yield from walk(c)

shape_per_region = df.groupby("region")["geo_shape_region"].agg(["first", "nunique"])
point_per_region = df.groupby("region")["geo_point_region"].agg(["first", "nunique"])
print("One shape per region:", bool((shape_per_region["nunique"] == 1).all()),
      "| one centroid per region:", bool((point_per_region["nunique"] == 1).all()))

geo_rows, features = [], []
for r, s in shape_per_region["first"].items():
    try:
        g = json.loads(s)
        pts = np.array(list(walk(g["coordinates"])))
        ok_parse = True
    except Exception as e:
        g, pts, ok_parse = None, np.empty((0, 2)), False
    try:
        lat, lon = [float(x) for x in str(point_per_region.loc[r, "first"]).split(",")]
    except Exception:
        lat = lon = np.nan
    in_france = bool(ok_parse and len(pts) and pts[:, 0].min() > -6 and pts[:, 0].max() < 10 and pts[:, 1].min() > 41 and pts[:, 1].max() < 52)
    code_r = df.loc[df.region == r, "code_insee_region"].iloc[0]
    geo_rows.append({"region": r, "code": code_r, "geom_type": g["type"] if g else None, "points": len(pts),
                     "lat": lat, "lon": lon, "parsed": ok_parse, "inside_France_bbox": in_france})
    if g:
        features.append({"type": "Feature", "id": code_r, "properties": {"region": r, "code": code_r}, "geometry": g})

geo_audit = pd.DataFrame(geo_rows)
display(geo_audit)
GEO_OK = bool(geo_audit["parsed"].all() and geo_audit["inside_France_bbox"].all() and geo_audit["lat"].notna().all())
print("Geometry valid for all regions:", GEO_OK)

if GEO_OK:
    (OUT_DIR / "regions.geojson").write_text(json.dumps({"type": "FeatureCollection", "features": features}), encoding="utf-8")
    print("Saved outputs/regions.geojson  (use it for the Plotly/pydeck choropleth - no external file needed)")
    issue("Info", "Geo", "geo_shape_region parses as valid GeoJSON for all 13 regions; centroids valid.",
          "Export `regions.geojson` + 13-row region table; drop the text columns from the fact table.")
else:
    issue("High", "Geo", "Geometry could not be validated for some regions.", "Fallback: join an external regions GeoJSON on INSEE code.")

## 8 · Energy source overview
**What:** build a tidy **long** table (`date, region, source, value`), apply the structural-zero rule for nuclear, then summarise each source: totals, share, typical monthly values, zeros and **negative values**; finally look at outliers **per region** and at the seasonal and annual patterns.
**Why:** this is the core of the story: which sources dominate, how big the renewable part is, and whether the numbers are plausible.
**Rule used for nuclear:** `NaN → 0` **only** for regions proven to have no nuclear plant in section 4 (documented assumption). Any other `NaN` stays missing.

In [ ]:
fact = df[["date", "region", "code_insee_region"] + MEASURES].copy()
# structural zero rule (data-driven: only regions that are NaN on all 144 months)
mask_struct = fact["region"].isin(NO_NUCLEAR)
fact.loc[mask_struct, NUC] = fact.loc[mask_struct, NUC].fillna(0.0)
print("Nuclear cells set to 0 (structural):", int(mask_struct.sum()), "| NaN left in measures:", int(fact[MEASURES].isna().sum().sum()))

long = fact.melt(id_vars=["date", "region", "code_insee_region"], value_vars=MEASURES, var_name="source", value_name="value")
long["family"] = long["source"].map(FAMILY)
print("Tidy table:", long.shape)

g = long.groupby("source")["value"]
overview = pd.DataFrame({
    "label": [LABEL[s] for s in g.sum().index],
    "family": long.groupby("source")["family"].first(),
    "total_GWh": g.sum(), "mean_GWh": g.mean(), "median_GWh": g.median(),
    "min_GWh": g.min(), "max_GWh": g.max(),
    "n_zero": g.apply(lambda s: int((s == 0).sum())), "n_negative": g.apply(lambda s: int((s < 0).sum())),
}).sort_values("total_GWh", ascending=False)
overview["share_%"] = (overview.total_GWh / overview.total_GWh.sum() * 100).round(2)
display(overview)

ren_total = overview.loc[overview.family == "Renewable", "total_GWh"].sum()
REN_CUM = ren_total / overview.total_GWh.sum() * 100
hydro_in_ren = overview.loc["production_hydraulique_gwh", "total_GWh"] / ren_total * 100
print(f"\nRenewable share of cumulated production 2014-2025: {REN_CUM:.1f}%")
print(f"Hydro represents {hydro_in_ren:.0f}% of all renewable production -> wind+solar+bio ('new renewables') = {100-hydro_in_ren:.0f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
o = overview.iloc[::-1]
ax.barh(o["label"], o["total_GWh"] / 1e3, color=[COLOR[s] for s in o.index])
for y, (v, p) in enumerate(zip(o["total_GWh"] / 1e3, o["share_%"])): ax.text(v + 30, y, f"{p:.1f}%", va="center", fontsize=9)
ax.set_xlabel("Cumulated production 2014-2025 (TWh = thousand GWh)"); ax.set_xlim(0, o["total_GWh"].max() / 1e3 * 1.12)
ax.set_title("Cumulated electricity production by source (13 metropolitan regions)")
plt.tight_layout(); plt.show()

**How to read this:** nuclear dominates (~70 %). Hydro is by far the largest renewable source (about half of all renewable GWh), then wind, solar and bioenergies. The storyline therefore needs to separate **hydro** (large, weather-dependent, almost no new capacity) from **wind + solar + bioenergies** (where the *transition* actually happens).

In [ ]:
# ---------- negative values: where and how big? ----------
neg = long[long["value"] < 0].copy()
ref = long.groupby(["region", "source"])["value"].median().rename("typical_GWh")
neg = neg.join(ref, on=["region", "source"])
neg["label"] = neg["source"].map(LABEL)
neg["size_vs_typical_%"] = (neg["value"] / neg["typical_GWh"] * 100).round(2)
print(f"{len(neg)} negative values  ->  by source:\n{neg.groupby('label').size().to_string()}\n")
display(neg.sort_values("value")[["date", "region", "label", "value", "typical_GWh", "size_vs_typical_%"]].head(40))
if len(neg):
    issue("Medium", "Validity", f"{len(neg)} negative production values ({', '.join(f'{k}: {v}' for k, v in neg.groupby('label').size().items())}); "
          f"smallest = {neg['value'].min():.3f} GWh.",
          "Keep raw values (they are net figures, e.g. plant self-consumption when offline), add flag `is_negative`, never delete; mention in limitations.")

In [ ]:
# ---------- outliers per REGION and source (3 x IQR) ----------
grp = long.dropna(subset=["value"]).groupby(["region", "source"])["value"]
q1, q3 = grp.transform(lambda s: s.quantile(.25)), grp.transform(lambda s: s.quantile(.75))
iqr = q3 - q1
long_o = long.dropna(subset=["value"]).assign(extreme=lambda x: (x.value < q1 - 3 * iqr) | (x.value > q3 + 3 * iqr))
out_tbl = long_o[long_o.extreme].groupby("source").size().rename("extreme_values_per_region_rule").to_frame()
out_tbl.index = [LABEL[s] for s in out_tbl.index]
print("Extreme values when each region is compared with ITSELF (3 x IQR):")
display(out_tbl if len(out_tbl) else "none")
print("Total flagged:", int(long_o.extreme.sum()), "of", len(long_o), f"values ({long_o.extreme.mean()*100:.2f}%)")
if long_o.extreme.sum():
    top = long_o[long_o.extreme].sort_values("value", ascending=False).head(8)
    display(top.assign(label=top["source"].map(LABEL))[["date", "region", "label", "value"]])
issue("Low", "Outliers", f"{int(long_o.extreme.sum())} extreme values with the per-region rule (run 1 global rule flagged 214 - mostly big regions, not errors).",
      "Keep all; use a flag `is_extreme` only to annotate charts. Growth in wind/solar and wet/dry hydro years explain most of them.")

In [ ]:
# ---------- annual mix, renewable share, hydro vs new renewables ----------
long["year"] = long["date"].dt.year
yr_src = long.pivot_table(index="year", columns="source", values="value", aggfunc="sum")[MEASURES]
yr_tot = yr_src.sum(axis=1)
yr = pd.DataFrame({
    "total_TWh": yr_tot / 1e3,
    "renewable_%": yr_src[[c for c in MEASURES if FAMILY[c] == "Renewable"]].sum(axis=1) / yr_tot * 100,
    "hydro_%": yr_src["production_hydraulique_gwh"] / yr_tot * 100,
    "new_renewables_%": yr_src[NEW_RENEWABLES].sum(axis=1) / yr_tot * 100,
    "nuclear_%": yr_src[NUC] / yr_tot * 100,
})
display(yr.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
bottom = np.zeros(len(yr_src))
for c in [NUC, "production_thermique_gwh", "production_hydraulique_gwh", "production_eolienne_gwh", "production_solaire_gwh", "production_bioenergies_gwh"]:
    axes[0].bar(yr_src.index, yr_src[c] / 1e3, bottom=bottom, color=COLOR[c], label=LABEL[c]); bottom += (yr_src[c] / 1e3).values
axes[0].set_title("Annual production by source"); axes[0].set_xlabel("Year"); axes[0].set_ylabel("Production (TWh per year)")
axes[0].legend(ncol=3, fontsize=8, loc="lower center"); axes[0].set_xticks(yr_src.index); axes[0].tick_params(axis="x", rotation=60)

axes[1].plot(yr.index, yr["renewable_%"], marker="o", color="#009E73", label="All renewables")
axes[1].plot(yr.index, yr["hydro_%"], marker="s", color="#0072B2", label="Hydro")
axes[1].plot(yr.index, yr["new_renewables_%"], marker="^", color="#D55E00", label="Wind + solar + bioenergies")
for col, c in [("renewable_%", "#009E73"), ("new_renewables_%", "#D55E00")]:
    axes[1].annotate(f"{yr[col].iloc[0]:.1f}%", (yr.index[0], yr[col].iloc[0]), textcoords="offset points", xytext=(0, 8), ha="center", color=c, fontsize=8)
    axes[1].annotate(f"{yr[col].iloc[-1]:.1f}%", (yr.index[-1], yr[col].iloc[-1]), textcoords="offset points", xytext=(0, 8), ha="center", color=c, fontsize=8)
axes[1].set_ylim(0, max(40, yr["renewable_%"].max() * 1.25)); axes[1].set_xlabel("Year"); axes[1].set_ylabel("Share of national production (%)")
axes[1].set_title("Share of national production (exploratory preview)"); axes[1].legend(fontsize=8); axes[1].set_xticks(yr.index); axes[1].tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

**How to read this:** left = the national mix (a plausibility check: totals stay in the order of magnitude of national electricity generation, and a dip is visible around 2022). Right = a *preview* of the story: the renewable share rises, and the split shows how much of that rise comes from wind + solar + bioenergies rather than hydro. These are **exploratory** numbers – the polished and annotated versions belong to the Streamlit app.

In [ ]:
# ---------- regional preview: first year vs last year ----------
reg_year = long.pivot_table(index=["region", "year"], columns="family", values="value", aggfunc="sum").fillna(0)
reg_year["total"] = reg_year.sum(axis=1)
reg_year["ren_%"] = reg_year["Renewable"] / reg_year["total"] * 100
first_y, last_y = int(long.year.min()), int(long.year.max())
cmp_ = pd.DataFrame({f"{first_y}_%": reg_year.xs(first_y, level="year")["ren_%"], f"{last_y}_%": reg_year.xs(last_y, level="year")["ren_%"]})
cmp_["delta_pts"] = cmp_[f"{last_y}_%"] - cmp_[f"{first_y}_%"]
cmp_["has_nuclear"] = ~cmp_.index.isin(NO_NUCLEAR)
cmp_ = cmp_.sort_values(f"{last_y}_%")
display(cmp_.round(1).sort_values(f"{last_y}_%", ascending=False))

fig, ax = plt.subplots(figsize=(9, 5.2))
y = np.arange(len(cmp_))
ax.hlines(y, cmp_[f"{first_y}_%"], cmp_[f"{last_y}_%"], color="#bbbbbb", lw=2)
ax.scatter(cmp_[f"{first_y}_%"], y, color="#7F7F7F", label=str(first_y), zorder=3)
ax.scatter(cmp_[f"{last_y}_%"], y, color="#009E73", label=str(last_y), zorder=3)
ax.set_yticks(y); ax.set_yticklabels(cmp_.index); ax.set_xlabel("Renewable share of regional production (%)")
ax.set_title(f"Renewable share by region, {first_y} vs {last_y} (exploratory)"); ax.legend()
plt.tight_layout(); plt.show()

**How to read this:** each line is a region; the gap between the two dots is its progress in percentage points. **Caution (limitation to state in the app):** this is a share of *regional production*, not of regional *consumption*. A region with little production (e.g. Île-de-France) or with a lot of nuclear (e.g. Centre-Val de Loire) has a very different baseline, so the dashboard should compare *progress* (points gained) and not only the level.

In [ ]:
# ---------- map feasibility check: choropleth drawn only from the file's own polygons ----------
if GEO_OK:
    share_last = cmp_[f"{last_y}_%"]
    cm = plt.get_cmap("YlGn"); vmin, vmax = share_last.min(), share_last.max()
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    for f in features:
        geom = f["geometry"]; polys = [geom["coordinates"]] if geom["type"] == "Polygon" else geom["coordinates"]
        val = share_last[f["properties"]["region"]]
        for poly in polys:
            ax.add_patch(MplPolygon(np.array(poly[0]), closed=True, facecolor=cm((val - vmin) / (vmax - vmin + 1e-9)), edgecolor="white", lw=0.8))
    allp = np.array([p for f in features for p in walk(f["geometry"]["coordinates"])])
    ax.set_xlim(allp[:, 0].min() - .3, allp[:, 0].max() + .3); ax.set_ylim(allp[:, 1].min() - .3, allp[:, 1].max() + .3)
    ax.set_aspect(1 / np.cos(np.radians(46.5))); ax.grid(False); ax.set_xticks([]); ax.set_yticks([])
    sm = plt.cm.ScalarMappable(cmap=cm, norm=plt.Normalize(vmin, vmax)); fig.colorbar(sm, ax=ax, shrink=.7, label=f"Renewable share {last_y} (%)")
    ax.set_title("Map feasibility check (polygons from the CSV itself)")
    plt.tight_layout(); plt.show()

In [ ]:
# ---------- seasonality (national, average calendar month) -> justifies seasonal features ----------
nat = long.groupby(["date", "source"])["value"].sum().unstack()
nat["month"] = nat.index.month
season = nat.groupby("month")[["production_solaire_gwh", "production_eolienne_gwh", "production_hydraulique_gwh"]].mean()
fig, axes = plt.subplots(1, 3, figsize=(13, 3.3), sharex=True)
for ax, c in zip(axes, season.columns):
    ax.bar(season.index, season[c], color=COLOR[c]); ax.set_title(LABEL[c]); ax.set_xlabel("Calendar month"); ax.set_xticks(range(1, 13))
axes[0].set_ylabel("Mean national production (GWh / month)")
fig.suptitle("Seasonal profile 2014-2025 (average over years)", y=1.03); plt.tight_layout(); plt.show()

peak = {LABEL[c]: int(season[c].idxmax()) for c in season.columns}
print("Peak month by source:", peak)

**How to read this:** solar peaks in summer, wind and hydro in winter/spring. Because of this strong seasonality, **month-on-month comparisons are misleading** → the app should compare the same month across years, or use 12-month rolling sums / annual shares.

## 9 · Data quality report
**What:** one scorecard (0–100) on six dimensions. *Completeness* is shown **raw** and **adjusted** (structural nuclear zeros are not errors).
**Why:** the rubric rewards *transparency about limitations*. This table can be reused as-is in the "Data Quality & Limitations" section of the app and in the demo video.

| Dimension | Definition |
|---|---|
| Completeness (raw) | non-empty cells ÷ all cells of the 11 original columns |
| Completeness (adjusted) | same, ignoring nuclear cells of regions proven to have no nuclear plant |
| Uniqueness | 1 − duplicated (month, region) ÷ rows |
| Validity | non-negative values ÷ non-empty production values |
| Consistency | code↔name one-to-one **and** equal to the official 13 regions |
| Temporal coverage | observed ÷ expected region-months |
| Geo validity | regions whose polygon parses and lies in France ÷ 13 |

In [ ]:
orig = list(SCHEMA)
comp_raw = 100 * (1 - df[orig].isna().sum().sum() / df[orig].size)
struct_cells = int(df.loc[df.region.isin(NO_NUCLEAR), NUC].isna().sum())
comp_adj = 100 * (1 - (df[orig].isna().sum().sum() - struct_cells) / (df[orig].size - struct_cells))
uniq = 100 * (1 - n_key / len(df))
vals = df[MEASURES].stack().dropna()
valid = 100 * (vals >= 0).mean()
consist = 100.0 if (code_name_ok and not unknown and not absent) else 50.0
cover = 100 * reg_t["observed"].sum() / reg_t["expected"].sum()
geo_v = 100 * (geo_audit["parsed"] & geo_audit["inside_France_bbox"]).mean()

scorecard = pd.DataFrame({"score": [comp_raw, comp_adj, uniq, valid, consist, cover, geo_v]},
                         index=["Completeness (raw)", "Completeness (adjusted)", "Uniqueness", "Validity", "Consistency", "Temporal coverage", "Geo validity"])
scorecard["status"] = pd.cut(scorecard.score, [-1, 80, 95, 100.0001], labels=["Poor", "Acceptable", "Good"])
OVERALL = scorecard.drop(index="Completeness (raw)").score.mean()      # raw completeness shown, adjusted one is scored
display(scorecard.round(2))
print(f"\nOVERALL QUALITY SCORE (adjusted): {OVERALL:.1f} / 100")

fig, ax = plt.subplots(figsize=(8, 3.8))
col = scorecard.score.map(lambda s: "#009E73" if s >= 95 else "#E69F00" if s >= 80 else "#D55E00")
ax.barh(scorecard.index[::-1], scorecard.score[::-1], color=col[::-1])
for i, v in enumerate(scorecard.score[::-1]): ax.text(min(v + 1, 88), i, f"{v:.1f}", va="center", fontsize=9, color="black")
ax.set_xlim(0, 100); ax.set_xlabel("Score (0-100)"); ax.set_title("Data quality scorecard")
plt.tight_layout(); plt.show()

## 10 · Reconciliation with run 1 and exports
**What:** (a) compare every key number of this notebook with the numbers you obtained in run 1 → `PASS/FAIL`; (b) save the artefacts needed by notebook 02 and by the Streamlit app.
**Why:** a result that cannot be reproduced is not evidence. If a `FAIL` ever appears, the CSV has changed (compare the SHA-256) or the code was modified.

In [ ]:
now = {
    "rows": len(df), "columns": RAW.shape[1], "regions": df.region.nunique(), "months": df.date.nunique(),
    "missing_cells": int(df[orig].isna().sum().sum()),
    "nuclear_total_gwh": overview.loc["production_nucleaire_gwh", "total_GWh"],
    "hydro_total_gwh": overview.loc["production_hydraulique_gwh", "total_GWh"],
    "thermal_total_gwh": overview.loc["production_thermique_gwh", "total_GWh"],
    "wind_total_gwh": overview.loc["production_eolienne_gwh", "total_GWh"],
    "solar_total_gwh": overview.loc["production_solaire_gwh", "total_GWh"],
    "bio_total_gwh": overview.loc["production_bioenergies_gwh", "total_GWh"],
    "neg_nuclear": int(overview.loc["production_nucleaire_gwh", "n_negative"]),
    "neg_thermal": int(overview.loc["production_thermique_gwh", "n_negative"]),
    "neg_wind": int(overview.loc["production_eolienne_gwh", "n_negative"]),
    "renewable_share_cum_pct": round(REN_CUM, 1),
}
rec = pd.DataFrame({"run1": pd.Series(RUN1), "now": pd.Series(now)})
rec["PASS"] = [abs(a - b) <= max(0.01, 1e-9 * abs(a)) for a, b in zip(rec.run1, rec.now)]
display(rec)
print(f"Reconciliation: {int(rec.PASS.sum())}/{len(rec)} checks PASS")
if not rec.PASS.all():
    print("-> Some numbers differ from run 1: check the SHA-256 and the file version.")

In [ ]:
# ---------- exports ----------
col_audit.to_csv(OUT_DIR / "column_audit.csv", index=False)
scorecard.to_csv(OUT_DIR / "quality_scorecard.csv")
overview.to_csv(OUT_DIR / "source_overview.csv")
yr.to_csv(OUT_DIR / "annual_preview.csv")

regions_ref = (geo_audit[["code", "region", "lat", "lon"]]
               .assign(has_nuclear=lambda x: ~x.region.isin(NO_NUCLEAR)).sort_values("code"))
regions_ref.to_csv(OUT_DIR / "region_reference.csv", index=False)

issues_df = pd.DataFrame(ISSUES, columns=["severity", "topic", "finding", "decision"])
order = {"High": 0, "Medium": 1, "Low": 2, "Info": 3}
issues_df = issues_df.sort_values("severity", key=lambda s: s.map(order)).reset_index(drop=True)
issues_df.to_csv(OUT_DIR / "issues.csv", index=False)

report = {
    "file": DATA_PATH.name, "sha256": SHA256, "encoding": ENCODING, "separator": SEPARATOR,
    "rows": int(len(df)), "columns": int(RAW.shape[1]), "layout": "wide",
    "unit": "GWh per region per month",
    "period": [str(start.date()), str(end.date())], "n_months": int(df.date.nunique()),
    "regions": sorted(df.region.unique()), "regions_without_nuclear": NO_NUCLEAR,
    "sources": [LABEL[s] for s in overview.index], "family_mapping": {LABEL[k]: v for k, v in FAMILY.items()},
    "renewable_share_cumulative_pct": round(float(REN_CUM), 2),
    "overall_quality_score": round(float(OVERALL), 1), "scorecard": scorecard.score.round(2).to_dict(),
    "reconciliation_pass": f"{int(rec.PASS.sum())}/{len(rec)}",
    "issues": issues_df.to_dict(orient="records"),
}
(OUT_DIR / "audit_report.json").write_text(json.dumps(report, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("Saved in", OUT_DIR.resolve())
for p in sorted(OUT_DIR.iterdir()): print("  -", p.name)

## 11 · Final synthesis
The next cells print the **dataset summary** and the **issues found**, built from the numbers computed above (nothing is typed by hand), followed by the recommendations and the engineered features.

In [ ]:
print("=" * 78); print("DATASET SUMMARY"); print("=" * 78)
print(f"File               : {DATA_PATH.name}  ({DATA_PATH.stat().st_size/1e6:.1f} MB, {ENCODING}, sep={SEPARATOR!r})")
print(f"Grain              : 1 row = 1 region x 1 month  ->  {len(df):,} rows = {df.region.nunique()} regions x {df.date.nunique()} months")
print(f"Period             : {start:%Y-%m} -> {end:%Y-%m}  (12 full years, balanced panel: {BALANCED})")
print(f"Measures           : 6 sources in GWh/month -> {', '.join(LABEL[c] for c in MEASURES)}")
print(f"Geography          : 13 metropolitan regions, polygons + centroids included (valid: {GEO_OK}); no overseas territories")
print(f"Regions w/o nuclear: {', '.join(NO_NUCLEAR) if NO_NUCLEAR else 'none'}")
print(f"Mix 2014-2025      : " + ", ".join(f"{LABEL[s]} {overview.loc[s, 'share_%']:.1f}%" for s in overview.index))
print(f"Renewable share    : {yr['renewable_%'].iloc[0]:.1f}% ({first_y}) -> {yr['renewable_%'].iloc[-1]:.1f}% ({last_y}); hydro = {hydro_in_ren:.0f}% of renewable GWh")
print(f"Quality score      : {OVERALL:.1f}/100  | reconciliation with run 1: {int(rec.PASS.sum())}/{len(rec)} PASS")
print(f"SHA-256            : {SHA256[:16]}...")
print()
print("=" * 78); print(f"ISSUES FOUND ({len(issues_df)})"); print("=" * 78)
display(issues_df)

### Issues found – reading guide
* **No blocking problem.** The dataset is clean, complete and well structured: balanced 13 × 144 panel, no duplicates, valid codes and geometry.
* The only "red flag" of run 1 (46 % missing nuclear) is **not an error**: it is a structural absence (regions without a nuclear plant) and is handled by a documented, data-driven rule.
* Real points of attention: 34 small negative values, a heavy geometry text repeated on every row, and **analytical limits** (below).

### Limitations to state in the app (transparency = graded)
1. **Production, not consumption**: shares are shares of *regional production*; they do not say how much of a region's own demand is renewable.
2. **Only 6 sources, metropolitan France only**; no installed capacity, no prices, no CO₂ → the story is about *production mix*, not cost or emissions.
3. **Hydro dominates renewables** and varies with rainfall; the transition signal sits mostly in wind + solar + bioenergies.
4. **Classification is an assumption**: hydro, wind, solar and bioenergies = renewable; thermal = fossil; nuclear = low-carbon but *not* renewable.
5. Strong **seasonality** (solar summer, wind/hydro winter): compare same months or annual / 12-month values.
6. Values are **net production** (small negatives exist); the data licence and exact definitions must be checked on the portal page and cited in the README.

### Recommendations for `02_cleaning_feature_engineering.ipynb` and `utils/prep.py`
| # | Action | Reason (finding) |
|---|---|---|
| 1 | Rename columns to `snake_case`, keep `ORIG_LABEL` for chart titles | accents/spaces |
| 2 | `date = pd.to_datetime(mois, format="%Y-%m")`; `code_insee_region` as 2-char text | text month, int code |
| 3 | **Split** into `fact_production` (region, date, source, GWh) and `dim_region` (code, region, lat, lon, has_nuclear) + `regions.geojson` | geo text = most of the 21 MB; faster app |
| 4 | Nuclear `NaN → 0` only for regions with 144/144 NaN; add `has_nuclear` | structural absence |
| 5 | Keep negatives, add `is_negative` flag; never delete | net production |
| 6 | Freeze `FAMILY` mapping in one module and show it in the app | classification is an assumption |
| 7 | Unit **GWh** on every axis, TWh for national totals | accessibility rubric |
| 8 | Save a parquet cache of the fact table; use `st.cache_data` | performance rubric |
| 9 | Re-run this audit's reconciliation after any data update (SHA-256) | reproducibility |
| 10 | Document source, retrieval date and licence in README | sourcing |

### Engineered features (for notebook 02)
| Feature | Definition | Story use |
|---|---|---|
| `year`, `month`, `quarter`, `season` | from `date` | seasonality, year filter |
| `total_gwh` | sum of the 6 sources (region, month) | denominator of every share |
| `renewable_gwh` | hydro + wind + solar + bio | headline KPI |
| `new_renewables_gwh` | wind + solar + bio | the "transition" signal, hydro excluded |
| `renewable_share_pct` | renewable / total × 100 | **main progress indicator per region** |
| `new_renewables_share_pct` | new renewables / total × 100 | growth story without hydro noise |
| `solar_share_pct`, `wind_share_pct` | source / total × 100 | links to your solar-technician hook |
| `low_carbon_share_pct` | (renewable + nuclear) / total | French context |
| `fossil_share_pct` | thermal / total | decarbonisation view |
| `renewable_12m_gwh` | 12-month rolling sum (region) | removes seasonality in trends |
| `yoy_change_pts` | share vs same month one year earlier | clean year-on-year change |
| `delta_share_pts` | share(2025) − share(2014), per region | **before / after** ranking |
| `cagr_new_renewables` | compound annual growth 2014→2025 (needs > 0 start) | "fastest-growing" ranking |
| `rank_region_year` | rank of renewable share per year | bump chart |
| `mix_diversity` | 1 − Σ(share²) (Herfindahl) | how balanced a region's mix is |
| `seasonal_index` | monthly value ÷ region-year mean | small multiples by season |
| `has_nuclear`, `is_negative`, `is_extreme` | flags | honest annotations |
| *(optional join)* regional population (INSEE) | GWh per inhabitant | fairness between big and small regions |
| *(optional join)* regional consumption (same ODRE portal) | production ÷ consumption | answers the "production vs demand" limitation |

### Next step
Run all cells. Check that the reconciliation shows **all PASS**, then send me `outputs/audit_report.json` (or a screenshot of sections 4, 9 and 10). I will then build `02_cleaning_feature_engineering.ipynb`.